# 🏃 Zonas de Entrenamiento en Running — Test de 5 Minutos

---

## Fundamento fisiológico

El **test de 5 minutos** permite estimar la **Velocidad Aeróbica Máxima (VAM)**: la velocidad mínima que solicita el VO₂máx.

$$VAM \; (km/h) = \frac{Distancia \; (m)}{5} \times \frac{60}{1000}$$

A partir de la VAM se generan **5 zonas de entrenamiento** en km/h, min/km y paso/400m.

---
### 📋 Instrucciones:
1. Ejecuta la **Celda 1** (instala librerías) y la **Celda 2** (define funciones)
2. Ejecuta la **Celda 3**: aparece el formulario interactivo
3. Ajusta los sliders y haz clic en **▶ Calcular zonas**

> 💡 **Concepto clave de programación:** En vez de editar variables directamente, usamos *widgets* — elementos de interfaz que capturan los valores del usuario sin necesidad de tocar el código.

In [ ]:
# ============================================================
# CELDA 1 — Instalación de librerías
# ============================================================
# 📖 En Python, 'pip install' descarga módulos externos.
# El '!' al inicio indica que es un comando de terminal, no Python.
# ipywidgets → interfaz interactiva (sliders, botones)
# plotly     → gráficos interactivos con zoom y hover
# pandas     → manejo de tablas (como Excel en Python)

!pip install ipywidgets plotly openpyxl --quiet

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print('✅ Librerías listas. Ejecuta la Celda 2.')

In [ ]:
# ============================================================
# CELDA 2 — Funciones de cálculo
# ============================================================
# 📖 CONCEPTO: def (definir funciones)
# Una función recibe parámetros, ejecuta operaciones y retorna resultados.
# Ventaja: defines la lógica UNA vez y la usas infinitas veces.
# El texto entre triple comillas (docstring) documenta qué hace la función.

def kmh_a_minkm(v):
    """Convierte velocidad km/h a ritmo min:seg/km."""
    if v <= 0: return '--'
    t = 60 / v
    return f'{int(t)}:{int((t - int(t)) * 60):02d}'

def kmh_a_paso400(v):
    """Calcula tiempo para recorrer 400m a velocidad v (km/h)."""
    if v <= 0: return '--'
    s = (3600 / v) * 0.4
    return f'{int(s//60)}:{int(s%60):02d}'

def ritmo_a_seg(r):
    """Convierte string 'M:SS' a segundos totales (para graficar)."""
    try:
        p = r.split(':')
        return int(p[0]) * 60 + int(p[1])
    except:
        return 0

# Tabla de zonas: (nombre, %VAM_min, %VAM_max, color_hex, descripcion)
# Modelo basado en Billat (2001) — 5 zonas
ZONAS_DEF = [
    ('Z1 — Regenerativa',  50,  65, '#74B9FF', 'Recuperación activa. FC baja. Sustrato principal: grasas.'),
    ('Z2 — Aeróbica base', 65,  78, '#55EFC4', 'Resistencia aeróbica. Umbral aeróbico inferior.'),
    ('Z3 — Tempo',         78,  88, '#FFEAA7', 'Umbral de lactato / VT2 (convencionalmente llamado umbral anaeróbico, término en revisión). Lactato ~4 mmol. 20–60 min sostenibles.'),
    ('Z4 — Sub-VAM',       88,  95, '#FDCB6E', 'Alto VO₂. Calidad aeróbica. Intervalos largos.'),
    ('Z5 — VAM / VO₂máx', 95, 105, '#FF7675', 'Máximo consumo O₂. Intervalos cortos (30-30).'),
]

def calcular_zonas(VAM_kmh):
    """Genera lista de diccionarios con cada zona calculada desde la VAM."""
    # 📖 List comprehension: forma compacta de crear listas con un for.
    # [ expresion for variable in iterable ]
    zonas = []
    for nombre, p_min, p_max, color, desc in ZONAS_DEF:
        v_min = VAM_kmh * p_min / 100
        v_max = VAM_kmh * p_max / 100
        zonas.append({
            'nombre': nombre, 'p_min': p_min, 'p_max': p_max,
            'v_min': round(v_min,1), 'v_max': round(v_max,1),
            'r_rapido': kmh_a_minkm(v_max), 'r_lento': kmh_a_minkm(v_min),
            'p400_rapido': kmh_a_paso400(v_max), 'p400_lento': kmh_a_paso400(v_min),
            'color': color, 'desc': desc
        })
    return zonas

print('✅ Funciones definidas:')
print('   kmh_a_minkm()   → convierte km/h a min:seg/km')
print('   kmh_a_paso400() → calcula tiempo en 400m')
print('   calcular_zonas() → genera las 5 zonas desde la VAM')
print('\n➡️  Ejecuta la Celda 3 para ver el formulario interactivo')

In [ ]:
# ============================================================
# CELDA 3 — 🎛️ FORMULARIO INTERACTIVO
# ============================================================
# 📖 CONCEPTO: Widgets y callbacks
#
# Un 'widget' es un elemento visual interactivo.
# Un 'callback' es una función que se ejecuta cuando ocurre un evento.
#
# El patrón central es:
#   1. Crear widgets (campos, sliders, botones)
#   2. Definir función que procesa los valores
#   3. Conectar la función al botón con .on_click()
#
# Así el alumno interactúa sin tocar el código.

estilos = {'description_width': '210px'}
ancho   = widgets.Layout(width='500px')

# --- Widgets de entrada ---
w_nombre = widgets.Text(
    value='Atleta 1',
    description='👤 Nombre:',
    style=estilos, layout=ancho
)
w_peso = widgets.BoundedIntText(
    value=65, min=30, max=150,
    description='⚖️ Peso (kg):',
    style=estilos, layout=ancho
)
w_dist = widgets.IntSlider(
    value=1350, min=600, max=2200, step=10,
    description='📏 Distancia en 5 min (m):',
    style=estilos, layout=ancho,
    continuous_update=False
)

# Preview en tiempo real al mover el slider
# 📖 .observe() ejecuta una función cada vez que cambia el valor del widget
w_prev = widgets.HTML()
def actualizar_prev(change):
    vam = (w_dist.value / 5) * 60 / 1000
    w_prev.value = (
        f'<div style="padding:6px 0;color:#0984e3;font-size:13px">'
        f'⚡ VAM estimada: <b>{vam:.1f} km/h</b> &nbsp;|&nbsp; '
        f'Ritmo VAM: <b>{kmh_a_minkm(vam)} min/km</b>&nbsp;|&nbsp; '
        f'Paso 400m: <b>{kmh_a_paso400(vam)}</b></div>'
    )
w_dist.observe(actualizar_prev, names='value')
actualizar_prev(None)

# Botones
btn_calc = widgets.Button(description='▶  Calcular zonas',
    button_style='success', layout=widgets.Layout(width='190px', height='36px'))
btn_exp = widgets.Button(description='💾 Exportar',
    button_style='info', layout=widgets.Layout(width='150px', height='36px'),
    disabled=True)

# Áreas de salida
out_res  = widgets.Output()
out_graf = widgets.Output()
out_exp  = widgets.Output()

estado = {}  # guardará los resultados para exportar

# ---- Callback principal ----
def on_calcular(b):
    nombre = w_nombre.value.strip() or 'Atleta'
    peso   = w_peso.value
    dist   = w_dist.value

    VAM_mmin = dist / 5
    VAM_kmh  = VAM_mmin * 60 / 1000
    vo2max   = round(VAM_mmin * 0.209 + 0.182, 1)
    zonas    = calcular_zonas(VAM_kmh)

    estado.update({'nombre': nombre, 'peso': peso, 'dist': dist,
                   'VAM_kmh': VAM_kmh, 'vo2max': vo2max, 'zonas': zonas})

    # --- Tabla HTML ---
    with out_res:
        clear_output(wait=True)
        filas_zona = ''.join([
            f'<tr style="background:{z["color"]}">'  
            f'<td style="padding:7px 10px;font-weight:bold">{z["nombre"]}</td>'
            f'<td style="padding:7px 10px;text-align:center">{z["p_min"]}–{z["p_max"]}%</td>'
            f'<td style="padding:7px 10px;text-align:center">{z["v_min"]} – {z["v_max"]}</td>'
            f'<td style="padding:7px 10px;text-align:center">{z["r_rapido"]} – {z["r_lento"]}</td>'
            f'<td style="padding:7px 10px;text-align:center">{z["p400_rapido"]} – {z["p400_lento"]}</td>'
            f'</tr>'
            for z in zonas
        ])
        display(HTML(f"""
        <div style="background:#dfe6e9;border-radius:10px;padding:14px;margin:8px 0;font-family:sans-serif">
          <b>📊 {nombre}</b> &nbsp;|&nbsp; Peso: {peso} kg &nbsp;|&nbsp;
          Dist. 5min: {dist} m &nbsp;|&nbsp;
          <b>VAM: {VAM_kmh:.1f} km/h</b> &nbsp;|&nbsp;
          Ritmo: {kmh_a_minkm(VAM_kmh)} min/km &nbsp;|&nbsp;
          VO₂máx est.: {vo2max} ml/kg/min
        </div>
        <table style="border-collapse:collapse;width:100%;font-size:13px">
          <tr style="background:#2d3436;color:white">
            <th style="padding:8px">Zona</th><th style="padding:8px">% VAM</th>
            <th style="padding:8px">km/h</th><th style="padding:8px">min/km</th>
            <th style="padding:8px">Paso /400m</th>
          </tr>
          {filas_zona}
        </table>
        """))

    # --- Gráfico Plotly ---
    with out_graf:
        clear_output(wait=True)
        fig = make_subplots(rows=1, cols=2,
            subplot_titles=[
                f'Velocidad (km/h) — VAM: {VAM_kmh:.1f} km/h',
                f'Ritmo (min/km) — VAM: {kmh_a_minkm(VAM_kmh)} min/km'
            ])

        for z in zonas:
            hover = (f"<b>{z['nombre']}</b><br>"
                     f"{z['p_min']}–{z['p_max']}% VAM<br>"
                     f"{z['v_min']}–{z['v_max']} km/h<br>"
                     f"Ritmo: {z['r_rapido']}–{z['r_lento']} min/km<br>"
                     f"400m: {z['p400_rapido']}–{z['p400_lento']}<br>"
                     f"<i>{z['desc']}</i><extra></extra>")

            fig.add_trace(go.Bar(
                x=[z['v_max']-z['v_min']], y=[z['nombre']],
                base=z['v_min'], orientation='h',
                marker_color=z['color'],
                marker_line_color='white', marker_line_width=2,
                hovertemplate=hover, showlegend=False,
                text=f"{z['v_min']}–{z['v_max']}", textposition='inside'
            ), row=1, col=1)

            s1 = ritmo_a_seg(z['r_rapido'])
            s2 = ritmo_a_seg(z['r_lento'])
            fig.add_trace(go.Bar(
                x=[s2-s1], y=[z['nombre']],
                base=s1, orientation='h',
                marker_color=z['color'],
                marker_line_color='white', marker_line_width=2,
                hovertemplate=f"<b>{z['nombre']}</b><br>Ritmo: {z['r_rapido']}–{z['r_lento']} min/km<extra></extra>",
                showlegend=False,
                text=f"{z['r_rapido']}–{z['r_lento']}", textposition='inside'
            ), row=1, col=2)

        fig.add_vline(x=VAM_kmh, line_dash='dash', line_color='#2d3436',
                      annotation_text=f'VAM {VAM_kmh:.1f}',
                      annotation_font_size=11, row=1, col=1)

        # Eje x panel derecho: formatear en min:seg
        ticks_s = list(range(180, 720, 30))
        fig.update_xaxes(tickvals=ticks_s,
                         ticktext=[f"{s//60}:{s%60:02d}" for s in ticks_s],
                         row=1, col=2)
        fig.update_layout(
            title=f'🏃 Zonas de Entrenamiento Running — {nombre}',
            height=400, plot_bgcolor='#f8f9fa',
            margin=dict(l=170, r=20, t=80, b=40)
        )
        fig.show()

    btn_exp.disabled = False

# ---- Callback exportar ----
def on_exportar(b):
    with out_exp:
        clear_output(wait=True)
        if not estado: print('⚠️ Primero calcula las zonas'); return
        ts = datetime.now().strftime('%Y%m%d_%H%M')
        fn = f"zonas_running_{estado['nombre'].replace(' ','_')}_{ts}"
        df = pd.DataFrame([{
            'Zona': z['nombre'], '%VAM': f"{z['p_min']}–{z['p_max']}%",
            'V_min_kmh': z['v_min'], 'V_max_kmh': z['v_max'],
            'Ritmo_rapido': z['r_rapido'], 'Ritmo_lento': z['r_lento'],
            'Paso400_rapido': z['p400_rapido'], 'Paso400_lento': z['p400_lento']
        } for z in estado['zonas']])
        df.to_csv(f'{fn}.csv', index=False)
        df.to_excel(f'{fn}.xlsx', index=False)
        print(f'✅ Exportado: {fn}.csv | {fn}.xlsx')
        print('📥 Descarga desde el panel 📁 izquierdo de Colab')

btn_calc.on_click(on_calcular)
btn_exp.on_click(on_exportar)

# --- Mostrar interfaz ---
display(HTML('<h3 style="color:#2d3436;font-family:sans-serif">📝 Ingresa los datos del atleta</h3>'))
display(w_nombre, w_peso, w_dist, w_prev)
display(widgets.HBox([btn_calc, btn_exp]))
display(out_res, out_graf, out_exp)

---
## 📚 Preguntas de reflexión

1. **Explora el slider:** Cambia la distancia de 900m a 1800m. ¿Cuánto varía la Z3 (Tempo) en min/km? ¿Qué implicancias prácticas tiene esto?

2. **Conversión de unidades:** ¿Por qué el ritmo en min/km y la velocidad en km/h son inversamente proporcionales? Escribe la fórmula.

3. **Código:** En la Celda 3, ¿qué hace la línea `w_dist.observe(actualizar_prev, names='value')`? ¿Por qué el preview se actualiza solo al mover el slider?

4. **Fisiología:** ¿Qué adaptaciones se buscan en Z2 vs Z4? ¿Por qué no conviene estar siempre en Z5?

5. **Aplicación:** Si dos atletas tienen el mismo peso pero uno corrió 1200m y otro 1600m, ¿cuál es la diferencia en su Z3 (km/h y min/km)? Usa el notebook para comparar.

---
*Notebook — Ciencias del Deporte | github.com/vgarrido1995*